# 07 - Template features, H1, and grouped feature ablations

**Influenza Season Forecasting** - Notebook 7 of 7

**Purpose:** Add the template lag, rolling, and hospitalization features; test H1 against the
incumbent cumulative-ILI model; and build D3 with standardized ridge coefficients plus grouped
block ablations. All results are preliminary and descriptive.

**Option B assumption, pending advisor decision.** Panel A uses the 19-season core period. Panel B
uses the 14-season 2009+ enrichment period. This two-panel structure presupposes Option B and does
not resolve the still-open Option A versus Option B decision.

Panel A has 7 conceptual features, 9 model columns, and n=19. Panel B has 9 conceptual features,
11 model columns, and n=14. Both ratios are strained. Panel B is high-variance descriptive
analysis and cannot support a feature-importance ranking.

## Setup and exact reconstruction of notebook 02 targets

This notebook deliberately repeats the established reconstruction used by notebooks 05 and 06.
The deferred `src/` refactor remains out of scope.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest

DATA_DIR = next((Path(p) for p in ["data/raw", "../data/raw"] if Path(p).exists()), Path("data/raw"))
RESULTS_DIR = DATA_DIR.parent.parent / "results"; RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = DATA_DIR.parent.parent / "figures"; FIG_DIR.mkdir(exist_ok=True)
EXCLUDED = {"2008-09", "2009-10", "2020-21"}
SPECIAL_CASES = {
    "2008-09": "pandemic-adjacent: spring 2009 H1N1 emergence wave",
    "2009-10": "2009 H1N1 pandemic season",
    "2020-21": "near-total flu absence under COVID NPIs",
}
DECISION_WEEKS = [8, 12, 16]
SEED = 42
BOOTSTRAP_N = 20000
np.random.seed(SEED)

def season_of(y, w):
    sy = y if w >= 40 else y - 1
    return f"{sy}-{str(sy + 1)[2:]}", sy

def sw(w):
    return w - 39 if w >= 40 else w + 13

# Rebuild weekly and season_table from notebook 02 logic, identical to notebooks 05 and 06.
ili = pd.read_csv(DATA_DIR / "ILINet.csv", skiprows=1, na_values=["X"])
_i = ili.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
ili["season"] = [x[0] for x in _i]; ili["ssy"] = [x[1] for x in _i]
ili["order"] = ili["WEEK"].apply(lambda w: w if w >= 40 else w + 100)
ili = ili.sort_values(["ssy", "order"]).reset_index(drop=True)
ili["sw"] = ili["WEEK"].apply(sw)

def _complete(g):
    sy = int(g["ssy"].iloc[0])
    sp = sorted(g.loc[g["YEAR"] == sy, "WEEK"])
    ep = sorted(g.loc[g["YEAR"] == sy + 1, "WEEK"])
    return bool(sp and sp[0] == 40 and ep and ep[0] == 1 and ep[-1] == 39)

weekly = ili[ili["season"].isin([s for s, g in ili.groupby("season") if _complete(g)])].copy()
rows = []
for s, g in weekly.groupby("season"):
    g = g.sort_values("order")
    sm3 = g["% WEIGHTED ILI"].rolling(3, center=True).mean()
    sm5 = g["% WEIGHTED ILI"].rolling(5, center=True).mean()
    rows.append(dict(season=s, ssy=int(g["ssy"].iloc[0]),
                     peak_week=int(g.loc[sm3.idxmax(), "WEEK"]),
                     peak_ili_pct=round(float(sm3.max()), 3),
                     peak_week_sm5=int(g.loc[sm5.idxmax(), "WEEK"])))
season_table = pd.DataFrame(rows).sort_values("ssy").reset_index(drop=True)
season_table["fragile_peak_week"] = season_table["peak_week"] != season_table["peak_week_sm5"]
season_table["sw_true"] = season_table["peak_week"].apply(sw)
EVAL = [s for s in season_table["season"] if s not in EXCLUDED]
ev = season_table[season_table["season"].isin(EVAL)].reset_index(drop=True)
assert len(EVAL) == 19 and int(ev["fragile_peak_week"].sum()) == 8
baseline04 = json.loads((RESULTS_DIR / "04_baseline_summary.json").read_text())
print("eval seasons:", len(EVAL), "| complete seasons:", len(season_table))
print("04 baselines:", [b["baseline"] for b in baseline04])

## Through-W features and the two availability concepts

For each fixed decision week W:

- `ili_lag_1..4` are ILI at season weeks W, W-1, W-2, and W-3.
- `ili_rolling4` is their arithmetic mean.
- `hosp_rate_lag1` is the FluSurv-NET weekly overall rate at the most recent indexed week at or
  before W.
- `cum_ili`, strain, and vaccine coverage retain notebook 06's through-W definitions.

**Index availability is not reporting availability.** The assertions below prove that no index
after W enters a feature. They do not prove that a value had been published by W. ILI is revised
but broadly near-real-time. FluSurv-NET, NREVSS strain, and FluVaxView are reporting-lagged or
survey-revised. Models containing hospitalization, strain, or vaccine features are therefore
retrospective and explanatory, not real-time forecasts.

In [ ]:
# NREVSS weekly strain buckets, using the established 2015-16 stitch.
def load_nrevss(f):
    d = pd.read_csv(DATA_DIR / f, skiprows=1, na_values=["X", "XX"])
    ii = d.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
    d["season"] = [x[0] for x in ii]; d["ssy"] = [x[1] for x in ii]
    d["sw"] = d["WEEK"].apply(sw)
    return d

def strain_buckets(d):
    col = lambda n: d[n] if n in d.columns else 0
    return pd.DataFrame({"season": d["season"], "ssy": d["ssy"], "sw": d["sw"],
        "A(H1N1)": col("A (H1)") + col("A (2009 H1N1)"),
        "A(H3N2)": col("A (H3)"),
        "B": col("B") + col("BVic") + col("BYam")})

comb = strain_buckets(load_nrevss("ICL_NREVSS_Combined_prior_to_2015_16.csv"))
phl = strain_buckets(load_nrevss("ICL_NREVSS_Public_Health_Labs.csv"))
strain_wk = pd.concat([comb[comb["ssy"] <= 2014], phl[phl["ssy"] >= 2015]]).reset_index(drop=True)

def dominant_strain_thruW(s, W):
    d = strain_wk[(strain_wk["season"] == s) & (strain_wk["sw"] <= W)]
    tot = d[["A(H1N1)", "A(H3N2)", "B"]].sum()
    return "none" if tot.sum() <= 0 else tot.idxmax()

# FluVaxView national all-ages coverage, matching notebook 06.
ALLAGES = [">=6 Months", "Greater than 6 Months flu"]
keep = []
for ch in pd.read_csv(DATA_DIR / "FluVaxView.csv", chunksize=200_000, dtype=str):
    m = ch[(ch["Geography"] == "United States") & (ch["Dimension Type"] == "Age") &
           (ch["Dimension"].isin(ALLAGES))]
    if len(m): keep.append(m)
if not keep:
    raise ValueError("No FluVaxView national all-ages rows matched; extend ALLAGES if the label changed.")
vax = pd.concat(keep); vax["est"] = pd.to_numeric(vax["Estimate (%)"], errors="coerce")
MONTH_SW = {9: -1, 10: 3, 11: 7, 12: 11, 1: 15, 2: 19, 3: 24, 4: 28, 5: 32, 6: 36, 7: 40, 8: 44}
vax["msw"] = pd.to_numeric(vax["Month"], errors="coerce").map(MONTH_SW)

# FluSurv-NET weekly overall hospitalization rate.
hosp = pd.read_csv(DATA_DIR / "FluSurveillance_Custom_Download_Data.csv", skiprows=2, na_values=["null"])
hosp.columns = [c.strip() for c in hosp.columns]
hosp = hosp[(hosp["CATCHMENT"] == "Entire Network") &
            (hosp["AGE CATEGORY"] == "Overall") &
            (hosp["SEX CATEGORY"] == "Overall") &
            (hosp["RACE CATEGORY"] == "Overall") &
            (hosp["VIRUS TYPE CATEGORY"] == "Overall")].copy()
hosp["season"] = hosp["YEAR"].astype(str)
hosp["WEEK"] = pd.to_numeric(hosp["WEEK"], errors="raise").astype(int)
hosp["sw"] = hosp["WEEK"].apply(sw)
hosp["order"] = hosp["WEEK"].apply(lambda w: w if w >= 40 else w + 100)
hosp["hosp_rate"] = pd.to_numeric(hosp["WEEKLY RATE"], errors="coerce")
assert not hosp.duplicated(["season", "YEAR.1", "WEEK"]).any()

# The established 52-week sw mapping gives week 53 and week 1 the same index (sw=14).
# For exact lag lookup, preserve the project convention and take the chronologically later row.
lag_collisions = weekly[weekly.duplicated(["season", "sw"], keep=False)]
collision_is_53_to_1 = lag_collisions.groupby(["season", "sw"])["WEEK"].apply(
    lambda values: set(values.astype(int)) == {1, 53})
assert len(collision_is_53_to_1) and collision_is_53_to_1.all()
weekly_lag = (weekly.sort_values(["season", "order"])
                    .drop_duplicates(["season", "sw"], keep="last"))
assert not weekly_lag.duplicated(["season", "sw"]).any()

def cum_ili_thruW(s, W):
    return float(weekly[(weekly["season"] == s) & (weekly["sw"] <= W)]["% WEIGHTED ILI"].sum())

def ili_lags(s, W):
    d = weekly_lag[weekly_lag["season"] == s].set_index("sw")["% WEIGHTED ILI"]
    return [float(d.loc[W - (k - 1)]) for k in range(1, 5)]

def vax_coverage_thruW(s, W):
    d = vax[(vax["Season/Survey Year"] == s) & vax["est"].notna() & (vax["msw"] <= W)]
    return float(d["est"].max()) if len(d) else np.nan

def hosp_rate_lag1(s, W):
    d = hosp[(hosp["season"] == s) & (hosp["sw"] <= W)].sort_values(["sw", "order"])
    return float(d.iloc[-1]["hosp_rate"]) if len(d) and pd.notna(d.iloc[-1]["hosp_rate"]) else np.nan

def feat_df(W, seasons):
    feature_rows = []
    peak_map = dict(zip(season_table["season"], season_table["peak_ili_pct"]))
    for s in seasons:
        lags = ili_lags(s, W)
        feature_rows.append(dict(season=s, cum_ili=cum_ili_thruW(s, W),
            ili_lag_1=lags[0], ili_lag_2=lags[1], ili_lag_3=lags[2], ili_lag_4=lags[3],
            ili_rolling4=float(np.mean(lags)), strain=dominant_strain_thruW(s, W),
            vax=vax_coverage_thruW(s, W), hosp_rate_lag1=hosp_rate_lag1(s, W),
            peak=float(peak_map[s])))
    return pd.DataFrame(feature_rows)

# Index firewall, distinct from reporting availability.
audit_pairs = 0
for W in DECISION_WEEKS:
    for s in season_table["season"]:
        g = weekly[weekly["season"] == s]
        g_lag = weekly_lag[weekly_lag["season"] == s]
        used = g[g["sw"] <= W]
        assert len(used) and int(used["sw"].max()) <= W
        for k in range(1, 5):
            lag_index = W - (k - 1)
            lag_rows = g_lag[g_lag["sw"] == lag_index]
            assert lag_index <= W and len(lag_rows) == 1
            assert int(lag_rows.iloc[0]["sw"]) <= W
        sd = strain_wk[(strain_wk["season"] == s) & (strain_wk["sw"] <= W)]
        if len(sd): assert int(sd["sw"].max()) <= W
        vd = vax[(vax["Season/Survey Year"] == s) & vax["est"].notna() & (vax["msw"] <= W)]
        if len(vd): assert float(vd["msw"].max()) <= W
        hd = hosp[(hosp["season"] == s) & (hosp["sw"] <= W)]
        if len(hd): assert int(hd["sw"].max()) <= W
        audit_pairs += 1
print(f"INDEX FIREWALL: all feature indices <= W across {audit_pairs} (season,W) pairs: PASS")
print("REPORTING AVAILABILITY: not guaranteed for FluSurv-NET, NREVSS, or FluVaxView; explanatory only")

## Two-panel ridge and corrected nested-CV imputation

The ridge structure follows notebook 06: standardized columns, nested lambda selection, and outer
LOSO. Notebook 07 deliberately corrects notebook 06's simpler imputation path. Every column with
missing values is imputed from its own training-fold mean. During inner CV, those means come only
from the inner-training rows and are applied to the inner-validation row. Notebook 06 is not
reopened because its protected path concerns one trailing vaccine column and only lambda selection.

In [ ]:
LAM_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]
PANEL_A_NAMES = ["cum_ili", "ili_lag_1", "ili_lag_2", "ili_lag_3", "ili_lag_4",
                 "ili_rolling4", "A(H1N1)", "A(H3N2)", "B"]
PANEL_B_NAMES = PANEL_A_NAMES + ["vax", "hosp_rate_lag1"]

def onehot_strain(series):
    cats = ["A(H1N1)", "A(H3N2)", "B"]
    return np.array([[1.0 if v == c else 0.0 for c in cats] for v in series])

def panel_a_matrix(F):
    return np.hstack([F[["cum_ili", "ili_lag_1", "ili_lag_2", "ili_lag_3",
                                "ili_lag_4", "ili_rolling4"]].values.astype(float),
                      onehot_strain(F["strain"])])

def panel_b_matrix(F):
    return np.hstack([panel_a_matrix(F), F[["vax", "hosp_rate_lag1"]].values.astype(float)])

def ridge_fit(X, y, lam):
    mu = X.mean(0); sd = X.std(0); sd[sd == 0] = 1.0
    Xs = (X - mu) / sd
    yb = y.mean()
    w = np.linalg.solve(Xs.T @ Xs + lam * np.eye(Xs.shape[1]), Xs.T @ (y - yb))
    return dict(w=w, b=yb, mu=mu, sd=sd)

def ridge_pred(model, X):
    return (X - model["mu"]) / model["sd"] @ model["w"] + model["b"]

def impute_train_rows(X_train, X_rows):
    train = X_train.copy(); rows = X_rows.copy()
    means = np.nanmean(train, axis=0)
    assert not np.isnan(means).any(), "an imputation column is entirely missing in the training fold"
    train_missing = np.where(np.isnan(train))
    train[train_missing] = means[train_missing[1]]
    row_missing = np.where(np.isnan(rows))
    rows[row_missing] = means[row_missing[1]]
    return train, rows, means

def select_lambda(X, y):
    scores = []
    for lam in LAM_GRID:
        errors = []
        for j in range(len(X)):
            mask = np.ones(len(X), bool); mask[j] = False
            inner_train, inner_val, _ = impute_train_rows(X[mask], X[j:j + 1])
            pred = ridge_pred(ridge_fit(inner_train, y[mask], lam), inner_val)[0]
            errors.append(abs(float(pred) - float(y[j])))
        scores.append(float(np.mean(errors)))
    return float(LAM_GRID[int(np.argmin(scores))])

def ridge_loso(X, y):
    preds, lambdas = [], []
    for i in range(len(X)):
        mask = np.ones(len(X), bool); mask[i] = False
        Xtr, ytr = X[mask], y[mask]
        lam = select_lambda(Xtr, ytr)
        outer_train, outer_test, _ = impute_train_rows(Xtr, X[i:i + 1])
        preds.append(float(ridge_pred(ridge_fit(outer_train, ytr, lam), outer_test)[0]))
        lambdas.append(lam)
    return np.array(preds), lambdas

def univariate_loso(x, y):
    preds = []
    for i in range(len(x)):
        mask = np.ones(len(x), bool); mask[i] = False
        b1, b0 = np.polyfit(x[mask], y[mask], 1)
        preds.append(float(b0 + b1 * x[i]))
    return np.array(preds)

def mae(pred, true):
    return float(np.mean(np.abs(np.asarray(pred) - np.asarray(true))))

def rmse(pred, true):
    return float(np.sqrt(np.mean((np.asarray(pred) - np.asarray(true)) ** 2)))

def baseline_c_severity(s, W):
    return float(weekly[(weekly["season"] == s) & (weekly["sw"] <= W)]["% WEIGHTED ILI"].max())

S2009 = [s for s in EVAL if not np.isnan(vax_coverage_thruW(s, 16))]
assert len(S2009) == 14
PANEL_CACHE = {}
panel_performance = []
for W in DECISION_WEEKS:
    FA = feat_df(W, EVAL); XA = panel_a_matrix(FA); yA = FA["peak"].values
    pA, lamA = ridge_loso(XA, yA)
    PANEL_CACHE[("A", W)] = dict(F=FA, X=XA, y=yA, pred=pA, lambdas=lamA)
    panel_performance.append(dict(panel="A", W=W, n=19, conceptual_features=7, model_columns=9,
        MAE=round(mae(pA, yA), 3), RMSE=round(rmse(pA, yA), 3),
        baselineC_MAE=round(mae([baseline_c_severity(s, W) for s in FA.season], yA), 3)))

    FB = feat_df(W, S2009); XB = panel_b_matrix(FB); yB = FB["peak"].values
    pB, lamB = ridge_loso(XB, yB)
    PANEL_CACHE[("B", W)] = dict(F=FB, X=XB, y=yB, pred=pB, lambdas=lamB)
    panel_performance.append(dict(panel="B", W=W, n=14, conceptual_features=9, model_columns=11,
        MAE=round(mae(pB, yB), 3), RMSE=round(rmse(pB, yB), 3),
        baselineC_MAE=round(mae([baseline_c_severity(s, W) for s in FB.season], yB), 3)))
panel_df = pd.DataFrame(panel_performance)
print("Option B assumption, pending advisor decision")
print(panel_df.to_string(index=False))

## H1: recent four-week ILI versus cumulative ILI

H1 states that ILI from the four weeks prior to prediction is the strongest predictor of peak
severity. At each W, a lags-only ridge (`ili_lag_1..4` plus `ili_rolling4`) is compared head to
head with notebook 06's cumulative-ILI-only LOSO regression, the 1.341 climatology floor, and
baseline C at the same W. `cum_ili` is not a template feature, so this also tests whether the
project's substitution was justified.

Inference is paired by season. The reported delta is `lags absolute error - cumulative-ILI
absolute error`, so negative favors the lag model. The exact two-sided sign test excludes ties.
The bootstrap interval is a percentile interval for the paired mean delta. Three decision weeks
are inspected, so results are descriptive support or no support, never confirmation.

In [ ]:
h1_summary, h1_paired = [], []
for W in DECISION_WEEKS:
    F = feat_df(W, EVAL); y = F["peak"].values.astype(float)
    X_lags = F[["ili_lag_1", "ili_lag_2", "ili_lag_3", "ili_lag_4", "ili_rolling4"]].values.astype(float)
    lag_pred, lag_lambdas = ridge_loso(X_lags, y)
    cum_pred = univariate_loso(F["cum_ili"].values.astype(float), y)
    clim_pred = np.array([float(y[np.arange(len(y)) != i].mean()) for i in range(len(y))])
    base_pred = np.array([baseline_c_severity(s, W) for s in F["season"]])
    lag_error = np.abs(lag_pred - y); cum_error = np.abs(cum_pred - y)
    delta = lag_error - cum_error
    lag_wins = int((delta < 0).sum()); cum_wins = int((delta > 0).sum()); ties = int((delta == 0).sum())
    non_ties = lag_wins + cum_wins
    sign_p = float(binomtest(lag_wins, non_ties, p=0.5, alternative="two-sided").pvalue) if non_ties else 1.0
    rng = np.random.default_rng(SEED + W)
    boot_idx = rng.integers(0, len(delta), size=(BOOTSTRAP_N, len(delta)))
    boot_mean = delta[boot_idx].mean(axis=1)
    ci_lo, ci_hi = np.quantile(boot_mean, [0.025, 0.975])
    h1_summary.append(dict(W=W, n=19, lags_MAE=round(mae(lag_pred, y), 3),
        cum_ili_MAE=round(mae(cum_pred, y), 3), climatology_MAE=round(mae(clim_pred, y), 3),
        baselineC_MAE=round(mae(base_pred, y), 3),
        mean_delta_lags_minus_cum=round(float(delta.mean()), 3),
        bootstrap_95_lo=round(float(ci_lo), 3), bootstrap_95_hi=round(float(ci_hi), 3),
        lag_wins=lag_wins, cum_ili_wins=cum_wins, ties=ties, sign_test_p=round(sign_p, 4)))
    for i, s in enumerate(F["season"]):
        h1_paired.append(dict(season=s, W=W, true_ili=round(float(y[i]), 3),
            lags_pred=round(float(lag_pred[i]), 4), cum_ili_pred=round(float(cum_pred[i]), 4),
            lags_abs_error=round(float(lag_error[i]), 4), cum_ili_abs_error=round(float(cum_error[i]), 4),
            delta_lags_minus_cum=round(float(delta[i]), 4)))
h1_df = pd.DataFrame(h1_summary)
h1_support = bool((h1_df["lags_MAE"] < h1_df["cum_ili_MAE"]).all())
if h1_support:
    h1_conclusion = ("Descriptive support for H1: the lag model has lower MAE at all three W values. "
                     "This is not confirmation because n=19 is small and three W comparisons were inspected.")
else:
    h1_conclusion = ("No descriptive support for H1: the lag model does not have lower MAE at all three W values. "
                     "The paired intervals and sign tests are descriptive at n=19 with three W comparisons.")
print(h1_df.to_string(index=False))
print("\n" + h1_conclusion)

## D3: standardized coefficients and grouped block ablations

`ili_rolling4` is exactly the mean of the four lag columns. Individual coefficients and
drop-one-lag deltas are therefore not feature rankings. D3 uses grouped block ablations as the
primary measure and reports standardized coefficients only as directional context.

Panel A has five ablation groups: cumulative ILI, the four-lag block, rolling summary, the
combined recent-ILI block (lags plus rolling), and strain. Panel B adds vaccine and
hospitalization for seven groups. Positive `delta_MAE` means removing the block worsened error.
Negative values mean the full model did not benefit from that block on this sample.

In [ ]:
PANEL_A_GROUPS = {
    "cum_ili": ["cum_ili"],
    "recent_ILI_lag_block": ["ili_lag_1", "ili_lag_2", "ili_lag_3", "ili_lag_4"],
    "rolling_summary": ["ili_rolling4"],
    "recent_ILI_all": ["ili_lag_1", "ili_lag_2", "ili_lag_3", "ili_lag_4", "ili_rolling4"],
    "strain_block": ["A(H1N1)", "A(H3N2)", "B"],
}
PANEL_B_GROUPS = {**PANEL_A_GROUPS, "vax": ["vax"], "hosp_rate_lag1": ["hosp_rate_lag1"]}

ridge_coefficients = []
ablation_records = []
for panel, names, groups in [("A", PANEL_A_NAMES, PANEL_A_GROUPS), ("B", PANEL_B_NAMES, PANEL_B_GROUPS)]:
    for W in DECISION_WEEKS:
        cache = PANEL_CACHE[(panel, W)]
        X, y = cache["X"], cache["y"]
        X_full, _, _ = impute_train_rows(X, np.empty((0, X.shape[1])))
        coef_model = ridge_fit(X_full, y, 1.0)
        for feature, value in zip(names, coef_model["w"]):
            ridge_coefficients.append(dict(panel=panel, W=W, feature=feature, coefficient=round(float(value), 6)))
        full_mae = mae(cache["pred"], y)
        for group, dropped in groups.items():
            keep_idx = [i for i, name in enumerate(names) if name not in dropped]
            ablated_pred, _ = ridge_loso(X[:, keep_idx], y)
            ablated_mae = mae(ablated_pred, y)
            ablation_records.append(dict(panel=panel, W=W, group=group,
                full_MAE=round(full_mae, 3), ablated_MAE=round(ablated_mae, 3),
                delta_MAE=round(ablated_mae - full_mae, 3)))
ablation_df = pd.DataFrame(ablation_records)
coef_df = pd.DataFrame(ridge_coefficients)
assert ablation_df[ablation_df.panel == "A"]["group"].nunique() == 5
assert ablation_df[ablation_df.panel == "B"]["group"].nunique() == 7
print(ablation_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 7.8), sharex=True)
colors = {8: "#4c78a8", 12: "#f58518", 16: "#54a24b"}
for ax, panel, n, columns, groups in [(axes[0], "A", 19, 9, PANEL_A_GROUPS),
                                        (axes[1], "B", 14, 11, PANEL_B_GROUPS)]:
    group_names = list(groups)
    y_pos = np.arange(len(group_names)); height = 0.22
    for offset, W in zip([-height, 0, height], DECISION_WEEKS):
        d = ablation_df[(ablation_df.panel == panel) & (ablation_df.W == W)].set_index("group")
        values = [float(d.loc[g, "delta_MAE"]) for g in group_names]
        ax.barh(y_pos + offset, values, height=height, color=colors[W], label=f"W={W}")
    ax.axvline(0, color="0.25", lw=1)
    ax.set_yticks(y_pos); ax.set_yticklabels(group_names, fontsize=8)
    ax.invert_yaxis(); ax.grid(axis="x", alpha=0.25)
    ax.set_xlabel("Ablated MAE minus full-model MAE")
    ax.set_title(f"Panel {panel}: {columns} columns, n={n}\nOption B assumption, pending advisor decision")
    c12 = coef_df[(coef_df.panel == panel) & (coef_df.W == 12)]
    coef_lines = [f"{r.feature}: {r.coefficient:+.3f}" for r in c12.itertuples()]
    split_at = (len(coef_lines) + 1) // 2
    coef_text = "W=12 standardized coefficients (lambda=1; directional only):\n"
    coef_text += "   ".join(coef_lines[:split_at]) + "\n" + "   ".join(coef_lines[split_at:])
    ax.text(0.0, -0.24, coef_text, transform=ax.transAxes, ha="left", va="top", fontsize=7, wrap=True)
axes[1].legend(loc="lower right", fontsize=8)
fig.suptitle("D3: grouped block ablations with standardized ridge coefficients", fontsize=15)
fig.text(0.5, 0.015,
    "Positive delta means the block improved LOSO MAE. Coefficients are directional only because rolling4 is an exact linear combination of the four lags. "
    "Panel A is strained at 9 columns on 19 seasons. Panel B is high-variance descriptive analysis at 11 columns on 14 seasons and is not ranking evidence.",
    ha="center", va="bottom", fontsize=8, color="0.3", wrap=True)
fig.tight_layout(rect=(0, 0.22, 1, 0.94))
fig.savefig(FIG_DIR / "11_D3_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved figures/11_D3_feature_importance.png")

## Excluded seasons under Panel A

The three excluded seasons are scored against a Panel A ridge trained on all 19 modeled seasons.
They never enter training or headline metrics. This is an excluded-from-training structural-break
stress test, not a prospective forecast: training includes seasons chronologically later than
2008-09 and 2009-10. Panel A also contains reporting-lagged strain, so it is explanatory. Results
are reported per season and are never pooled.

In [ ]:
special_records = []
for W in DECISION_WEEKS:
    train = PANEL_CACHE[("A", W)]
    FS = feat_df(W, list(SPECIAL_CASES)); XS = panel_a_matrix(FS)
    selected_lambda = select_lambda(train["X"], train["y"])
    X_train, X_special, _ = impute_train_rows(train["X"], XS)
    model = ridge_fit(X_train, train["y"], selected_lambda)
    pred = ridge_pred(model, X_special)
    for i, s in enumerate(FS["season"]):
        true = float(FS.iloc[i]["peak"]); estimate = float(pred[i])
        special_records.append(dict(season=s, mechanism=SPECIAL_CASES[s], W=W,
            model="Panel A ridge", selected_lambda=selected_lambda, true_ili=round(true, 3),
            pred_ili=round(estimate, 4), error=round(estimate - true, 4),
            abs_error=round(abs(estimate - true), 4),
            label="excluded-from-training structural-break stress test, not prospective"))
special_df = pd.DataFrame(special_records).sort_values(["season", "W"]).reset_index(drop=True)
print(special_df.to_string(index=False))

## Persisted results, interpretations, and template deviations

Two template deviations are narrow and explicit:

1. `season_week` is not usable as a season-level predictor at a fixed W because it equals W for
   every season and has zero variance. It remains the within-season time axis (`sw`) used by
   trajectory models, feature indexing, and figures.
2. ARIMA produces no prediction intervals in this implementation. D1 and interval coverage are
   therefore Prophet-only.

In [ ]:
def to_md(df):
    cols = list(df.columns)
    head = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join("---" for _ in cols) + " |"
    body = ["| " + " | ".join("" if pd.isna(v) else str(v) for v in row) + " |"
            for row in df.itertuples(index=False)]
    return "\n".join([head, sep] + body)

interpretive_lines = []
for panel in ["A", "B"]:
    d12 = ablation_df[(ablation_df.panel == panel) & (ablation_df.W == 12)]
    strongest = d12.loc[d12["delta_MAE"].idxmax()]
    interpretive_lines.append(
        f"At W=12 in Panel {panel}, the largest positive block-ablation delta is {strongest.group} "
        f"at {strongest.delta_MAE:+.3f} MAE. This is descriptive, not a feature ranking.")

payload = {
    "status": "preliminary descriptive analysis; Option B assumption, pending advisor decision",
    "decision_weeks": DECISION_WEEKS,
    "panel_definitions": {
        "A": {"n": 19, "conceptual_features": 7, "model_columns": 9,
              "caveat": "strained ratio; retrospective because strain is reporting-lagged"},
        "B": {"n": 14, "conceptual_features": 9, "model_columns": 11,
              "caveat": "high-variance descriptive analysis, not feature-ranking evidence"},
    },
    "panel_performance": panel_performance,
    "ridge_coefficients": ridge_coefficients,
    "block_ablations": ablation_records,
    "h1": {"hypothesis": "four-week prior ILI is the strongest predictor of peak severity",
            "conclusion": h1_conclusion, "summary": h1_summary, "paired_errors": h1_paired,
            "multiple_comparison_caveat": "three W values inspected; descriptive, never confirmation"},
    "special_cases_panel_A": {
        "analysis_type": "excluded-from-training structural-break stress test, not prospective",
        "records": special_records,
    },
    "firewall": {
        "index_firewall": f"PASS across {audit_pairs} (season,W) pairs; every feature index <= W",
        "reporting_availability": "not guaranteed for FluSurv-NET, NREVSS, or FluVaxView",
    },
    "imputation": "column-wise train means recomputed inside every inner-CV split; deliberate divergence from notebook 06",
    "template_deviations": {
        "season_week": "zero variance as a season-level predictor at fixed W; retained as within-season time axis sw",
        "ARIMA_intervals": "not implemented; D1 and interval coverage are Prophet-only",
    },
}
(RESULTS_DIR / "07_features_summary.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

md = [
    "# 07 template features, H1, and D3", "",
    "**Status:** Preliminary descriptive analysis. **Option B assumption, pending advisor decision.**", "",
    "## Availability and firewall", "",
    f"Index firewall: PASS across {audit_pairs} (season,W) pairs. Every feature index is <= W.", "",
    "This does not establish reporting availability. FluSurv-NET, NREVSS, and FluVaxView are lagged or revised,",
    "so models containing hospitalization, strain, or vaccine features are retrospective and explanatory.", "",
    "## Panel performance", "", to_md(panel_df), "",
    "Panel A has 9 columns on 19 seasons, a strained ratio. Panel B has 11 columns on 14 seasons and is",
    "high-variance descriptive analysis, not evidence for a feature-importance ranking.", "",
    "## H1 paired comparison", "", to_md(h1_df), "", h1_conclusion, "",
    "Negative paired deltas favor the lags-only model. Sign tests are exact and two-sided; bootstrap",
    "intervals cover the paired mean delta. Three W values were inspected, so no result confirms H1.", "",
    "### Per-season paired errors", "", to_md(pd.DataFrame(h1_paired)), "",
    "## Standardized ridge coefficients", "",
    "Coefficients are directional context only. `ili_rolling4` is exactly the mean of the four lag columns.", "",
    to_md(coef_df), "",
    "## Grouped block ablations", "",
    "Positive delta_MAE means removing the block worsened LOSO error. Grouped ablations are the primary",
    "importance measure because individual lag coefficients and drop-one-lag deltas are not interpretable.", "",
    to_md(ablation_df), "", *interpretive_lines, "",
    "## Excluded seasons under Panel A", "",
    "Excluded-from-training structural-break stress test, not a prospective forecast. Results are per season, never pooled.", "",
    to_md(special_df), "",
    "## Deliberate imputation divergence from notebook 06", "",
    "Notebook 07 imputes every missing column using its own training-fold mean and recomputes those means inside",
    "each inner-CV split. Notebook 06's simpler last-column, outer-fold path remains protected and unchanged.", "",
    "## Template deviations", "",
    "- `season_week` has zero variance as a season-level predictor at fixed W. It remains the within-season time axis `sw`.",
    "- ARIMA has no prediction intervals in this implementation. D1 and interval coverage are Prophet-only.", "",
]
(RESULTS_DIR / "07_features_summary.md").write_text("\n".join(md), encoding="utf-8")
print("saved results/07_features_summary.json and .md")

## Conclusion

Notebook 07 adds the template lag, rolling, and hospitalization features under an explicit index
firewall; separates that firewall from reporting availability; tests H1 with paired uncertainty;
and reports grouped block ablations instead of invalid individual importance rankings. The two
feature panels remain provisional under the unresolved Option B decision, and their column-count
to sample-size ratios limit interpretation.